In [35]:
from rdms_sector import (
    index_to_spin,
    spin_to_index,
    generate_binaries,
    _density_matrix_element,
    _hamming_weight, _binom, _get_index, make_binom_cache, density_matrix
)
import numpy as np
import lattice_symmetries as ls
import numpy.typing as npt
from spin_lattices import SquareLatticeNoDiag
from spin_systems import spin_system, heisenberg, no_symmetries_basis
import scipy
import scipy.linalg
from misc_utils import get_json_for_basis
import json
import itertools

In [2]:
def move_sites_to_back(
    wavefunction: npt.NDArray[np.complex128], basis: ls.Basis, sites: list[int]
) -> npt.NDArray[np.complex128]:
    unpacked_states = index_to_spin(basis.states, basis.number_sites)
    permuted_sites = sorted(set(range(basis.number_sites)) - set(sites)) + sites
    assert set(permuted_sites) == set(range(basis.number_sites))
    assert len(permuted_sites) == basis.number_sites
    permuted_unpacked_states = unpacked_states[:, np.argsort(permuted_sites)]
    permuted_states = spin_to_index(permuted_unpacked_states, basis.number_sites)
    return wavefunction[basis.index(permuted_states)]

In [3]:
basis = ls.SpinBasis(5, 3)
basis.build()
wavefunction = np.arange(len(basis.states), dtype=np.complex128)
assert (move_sites_to_back(wavefunction, basis, [4]) == wavefunction).all()
assert (move_sites_to_back(wavefunction, basis, [3, 4]) == wavefunction).all()
for s0, s1, s2, s3, s4 in index_to_spin(basis.states, 5):
    assert (
        wavefunction[basis.index(spin_to_index(np.array([[s0, s1, s2, s3, s4]]), 5))]
        == move_sites_to_back(wavefunction, basis, [3])[
            basis.index(spin_to_index(np.array([[s0, s1, s2, s4, s3]]), 5))
        ]
    )

for s0, s1, s2, s3, s4 in index_to_spin(basis.states, 5):
    assert (
        wavefunction[basis.index(spin_to_index(np.array([[s0, s1, s2, s3, s4]]), 5))]
        == move_sites_to_back(wavefunction, basis, [2])[
            basis.index(spin_to_index(np.array([[s0, s1, s3, s4, s2]]), 5))
        ]
    )

for s0, s1, s2, s3, s4 in index_to_spin(basis.states, 5):
    assert (
        wavefunction[basis.index(spin_to_index(np.array([[s0, s1, s2, s3, s4]]), 5))]
        == move_sites_to_back(wavefunction, basis, [1, 3])[
            basis.index(spin_to_index(np.array([[s0, s2, s4, s1, s3]]), 5))
        ]
    )

for s0, s1, s2, s3, s4 in index_to_spin(basis.states, 5):
    assert (
        wavefunction[basis.index(spin_to_index(np.array([[s0, s1, s2, s3, s4]]), 5))]
        == move_sites_to_back(wavefunction, basis, [3, 1])[
            basis.index(spin_to_index(np.array([[s0, s2, s4, s3, s1]]), 5))
        ]
    )

for s0, s1, s2, s3, s4 in index_to_spin(basis.states, 5):
    assert (
        wavefunction[basis.index(spin_to_index(np.array([[s0, s1, s2, s3, s4]]), 5))]
        == move_sites_to_back(wavefunction, basis, [2, 3])[
            basis.index(spin_to_index(np.array([[s0, s1, s4, s2, s3]]), 5))
        ]
    )

In [4]:
def reduced_density_matrix_blocks(
    wavefunction: npt.NDArray[np.complex128], basis: ls.Basis, sites: list[int]
) -> tuple[list[npt.NDArray[np.complex128]], list[npt.NDArray[np.uint64]]]:
    if basis.hamming_weight is None:
        raise ValueError("Only bases with a fixed hamming weight are supported.")
    if basis.symmetries != []:
        raise ValueError("Symmetries are not supported (yet)")
    if basis.spin_inversion is not None:
        raise ValueError("Spin inversion is not supported (yet)")
    
    hamming_weight = basis.hamming_weight
    n_sites = basis.number_sites
    wavefunction = move_sites_to_back(wavefunction, basis, sites)
    return density_matrix(len(sites), n_sites, hamming_weight, wavefunction)

In [5]:
def reduced_density_matrix(
    wavefunction: npt.NDArray[np.complex128],
    basis: ls.Basis,
    sites: list[int],
    return_states: bool = True,
) -> (
    npt.NDArray[np.complex128]
    | tuple[npt.NDArray[np.complex128], npt.NDArray[np.uint64]]
):
    matrices, sector_basis_states = reduced_density_matrix_blocks(
        wavefunction, basis, sites
    )
    if return_states:
        return scipy.linalg.block_diag(*matrices), np.concatenate(sector_basis_states)

    all_states = np.arange(2 ** len(sites), dtype=np.uint64)
    full_matrix = np.zeros((2 ** len(sites), 2 ** len(sites)), dtype=np.complex128)
    for matrix, states in zip(matrices, sector_basis_states):
        state_idxs = np.searchsorted(all_states, states)
        full_matrix[np.ix_(state_idxs, state_idxs)] = matrix
    return full_matrix

In [112]:
lattice = SquareLatticeNoDiag(4, 4, enumerate_along='x')
system = spin_system(heisenberg(lattice), no_symmetries_basis(hamming_weight='half'))
system_no_hamming = spin_system(heisenberg(lattice), no_symmetries_basis(hamming_weight=None))

In [113]:
expr_json = system.hamiltonian.expression.to_json()
basis_json = get_json_for_basis(system.basis)

In [114]:
basis_no_hamming_json = get_json_for_basis(system_no_hamming.basis)

In [129]:
expr_str = json.loads(expr_json)["expression"]
number_sites = system.basis.number_sites


In [131]:
print(f"{expr_str=}")
print(f"{number_sites=}")

expr_str='σᶻ₀ σᶻ₁ + σᶻ₀ σᶻ₃ + σᶻ₀ σᶻ₄ + σᶻ₀ σᶻ₁₂ + 2.0 σ⁺₀ σ⁻₁ + 2.0 σ⁺₀ σ⁻₃ + 2.0 σ⁺₀ σ⁻₄ + 2.0 σ⁺₀ σ⁻₁₂ + 2.0 σ⁻₀ σ⁺₁ + 2.0 σ⁻₀ σ⁺₃ + 2.0 σ⁻₀ σ⁺₄ + 2.0 σ⁻₀ σ⁺₁₂ + σᶻ₁ σᶻ₂ + σᶻ₁ σᶻ₅ + σᶻ₁ σᶻ₁₃ + 2.0 σ⁺₁ σ⁻₂ + 2.0 σ⁺₁ σ⁻₅ + 2.0 σ⁺₁ σ⁻₁₃ + 2.0 σ⁻₁ σ⁺₂ + 2.0 σ⁻₁ σ⁺₅ + 2.0 σ⁻₁ σ⁺₁₃ + σᶻ₂ σᶻ₃ + σᶻ₂ σᶻ₆ + σᶻ₂ σᶻ₁₄ + 2.0 σ⁺₂ σ⁻₃ + 2.0 σ⁺₂ σ⁻₆ + 2.0 σ⁺₂ σ⁻₁₄ + 2.0 σ⁻₂ σ⁺₃ + 2.0 σ⁻₂ σ⁺₆ + 2.0 σ⁻₂ σ⁺₁₄ + σᶻ₃ σᶻ₇ + σᶻ₃ σᶻ₁₅ + 2.0 σ⁺₃ σ⁻₇ + 2.0 σ⁺₃ σ⁻₁₅ + 2.0 σ⁻₃ σ⁺₇ + 2.0 σ⁻₃ σ⁺₁₅ + σᶻ₄ σᶻ₅ + σᶻ₄ σᶻ₇ + σᶻ₄ σᶻ₈ + 2.0 σ⁺₄ σ⁻₅ + 2.0 σ⁺₄ σ⁻₇ + 2.0 σ⁺₄ σ⁻₈ + 2.0 σ⁻₄ σ⁺₅ + 2.0 σ⁻₄ σ⁺₇ + 2.0 σ⁻₄ σ⁺₈ + σᶻ₅ σᶻ₆ + σᶻ₅ σᶻ₉ + 2.0 σ⁺₅ σ⁻₆ + 2.0 σ⁺₅ σ⁻₉ + 2.0 σ⁻₅ σ⁺₆ + 2.0 σ⁻₅ σ⁺₉ + σᶻ₆ σᶻ₇ + σᶻ₆ σᶻ₁₀ + 2.0 σ⁺₆ σ⁻₇ + 2.0 σ⁺₆ σ⁻₁₀ + 2.0 σ⁻₆ σ⁺₇ + 2.0 σ⁻₆ σ⁺₁₀ + σᶻ₇ σᶻ₁₁ + 2.0 σ⁺₇ σ⁻₁₁ + 2.0 σ⁻₇ σ⁺₁₁ + σᶻ₈ σᶻ₉ + σᶻ₈ σᶻ₁₁ + σᶻ₈ σᶻ₁₂ + 2.0 σ⁺₈ σ⁻₉ + 2.0 σ⁺₈ σ⁻₁₁ + 2.0 σ⁺₈ σ⁻₁₂ + 2.0 σ⁻₈ σ⁺₉ + 2.0 σ⁻₈ σ⁺₁₁ + 2.0 σ⁻₈ σ⁺₁₂ + σᶻ₉ σᶻ₁₀ + σᶻ₉ σᶻ₁₃ + 2.0 σ⁺₉ σ⁻₁₀ + 2.0 σ⁺₉ σ⁻₁₃ + 2.0 σ⁻₉ σ⁺₁₀ + 2.0 σ⁻₉ σ⁺₁₃ + σᶻ₁₀ σᶻ₁₁ 

In [130]:
basis = ls.SpinBasis(
    number_spins=number_sites,
    hamming_weight=number_sites // 2,
)
basis_no_hamming = ls.SpinBasis(
    number_spins=number_sites, hamming_weight=None
)
basis.build()
basis_no_hamming.build()
expression = ls.Expr(expr_str, particle="spin-1/2")
hamiltonian = ls.Operator(expression, basis)
hamiltonian_no_hamming = ls.Operator(expression, basis_no_hamming)
ground_state = scipy.sparse.linalg.eigsh(hamiltonian, k=1, which="SA")[1].ravel()
ground_state_no_hamming = scipy.sparse.linalg.eigsh(
    hamiltonian_no_hamming, k=1, which="SA"
)[1].ravel()

In [122]:
def rdms_reference(
    ground_state: npt.NDArray[np.float64], sites: list[int], n_spins: int
) -> npt.NDArray[np.float64]:
    ground_state_reshaped = ground_state.reshape((2,) * n_spins).transpose(
        range(n_spins - 1, -1, -1)
    )
    idxs = tuple(sorted(set(range(n_spins)) - set(sites)))
    ground_state_transposed = ground_state_reshaped.transpose(tuple(reversed(sites)) + idxs)
    ground_state_transposed = ground_state_transposed.reshape(
        (2 ** len(sites), 2 ** (n_spins - len(sites)))
    )
    return ground_state_transposed @ ground_state_transposed.T

In [123]:
np.set_printoptions(suppress=True)


In [126]:
for length in [1, 2, 3]:
    for sites in itertools.combinations(range(basis.number_sites), length):
        reference = rdms_reference(
            ground_state_no_hamming, list(sites), basis.number_sites
        )
        rdm = reduced_density_matrix(
            ground_state.astype(np.complex128), basis, list(sites), return_states=False
        )
        assert np.allclose(reference, rdm)

In [125]:
ground_state_no_hamming.reshape((2,) * 4).transpose((3, 2, 1, 0))[
    0, 0, 1, :
] @ ground_state_no_hamming.reshape((2,) * 4).transpose((3, 2, 1, 0))[0, 0, 1, :]

ValueError: cannot reshape array of size 65536 into shape (2,2,2,2)

In [33]:
lattice = SquareLatticeNoDiag(4, 4, enumerate_along='x')
system = spin_system(heisenberg(lattice), no_symmetries_basis(hamming_weight=2))
a, _ = reduced_density_matrix(system.ground_state.astype(np.complex128), system.basis, [0, 1, 2], return_states=True)

2024-07-03 21:56:29.208 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.


In [43]:
lattice = SquareLatticeNoDiag(4, 4, enumerate_along="x")
system = spin_system(heisenberg(lattice), no_symmetries_basis(hamming_weight=2))
b = reduced_density_matrix(
    system.ground_state.astype(np.complex128),
    system.basis,
    [0, 1, 2],
    return_states=False,
)[[0, 1, 2, 4, 3, 5, 6]][:, [0, 1, 2, 4, 3, 5, 6]]

2024-07-03 21:57:33.248 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.


In [45]:
np.allclose(a, b)

True